# Reproduce Key Results: Distance Ladder Systematics & Hubble Tension

**Interactive notebook to reproduce key findings from:**  
*Forensic Analysis of Distance Ladder Systematics: The Hubble Tension Reduced from ~6σ to ~1σ*

**Author:** Aaron Wiley  
**Journal:** The Astrophysical Journal (submitted)  
**Repository:** https://github.com/ylecoyote/distance-ladder-systematics

---

## Overview

This notebook demonstrates the main findings:

1. **Systematic Error Reassessment**: SH0ES underestimates σ_sys by 1.6×
2. **Tension Reduction**: From 5.9σ → 1.1σ through realistic error accounting
3. **JWST Cross-Validation**: Cepheid scatter 2.3× larger than TRGB-JAGB baseline
4. **Multi-Method Convergence**: Independent methods converge at H₀ ≈ 68 km/s/Mpc

**Runtime:** ~1-2 minutes

---

## Environment Setup

**Note:** This cell automatically detects your environment and sets up the necessary files.

- **Google Colab:** Clones the repository from GitHub
- **Binder/Local:** Uses existing files

**Run this cell first!**

In [ ]:
# Environment Setup: Clone repository if running in Google Colab
import os
import sys

try:
    # Check if running in Google Colab
    import google.colab
    IN_COLAB = True
    print("🔍 Detected Google Colab environment")
except ImportError:
    IN_COLAB = False
    print("🔍 Detected local or Binder environment")

if IN_COLAB:
    # Clone repository if not already present
    if not os.path.exists('distance-ladder-systematics'):
        print("📥 Cloning repository from GitHub...")
        !git clone https://github.com/ylecoyote/distance-ladder-systematics.git
        print("✓ Repository cloned successfully")
    else:
        print("✓ Repository already present")
    
    # Change to repository directory
    os.chdir('distance-ladder-systematics')
    print(f"✓ Changed to directory: {os.getcwd()}")
    
    # Verify data directory exists
    if os.path.exists('data'):
        print(f"✓ Data directory found with {len(os.listdir('data'))} files")
    else:
        print("⚠️ Warning: Data directory not found!")
else:
    # For Binder or local: assume we're already in the right directory
    if os.path.exists('data'):
        print(f"✓ Data directory found with {len(os.listdir('data'))} files")
        print(f"✓ Current directory: {os.getcwd()}")
    else:
        print("⚠️ Warning: Please ensure you're in the repository root directory")

print("\n" + "="*70)
print("Environment setup complete. Proceed to next cell.")
print("="*70)

---

## Setup: Import Libraries and Load Data

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Configure matplotlib for high-quality figures
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11
plt.rcParams['font.family'] = 'sans-serif'

# Set seaborn style
sns.set_style('whitegrid')

print("✓ Libraries imported successfully")
print(f"✓ NumPy version: {np.__version__}")
print(f"✓ Pandas version: {pd.__version__}")
print(f"✓ Matplotlib version: {plt.matplotlib.__version__}")

### Load Key Data Files

All CSV files include header comments documenting sources and methods.

In [ ]:
# Load data files (comment='#' skips header comments)
systematic_budget = pd.read_csv('data/systematic_error_budget.csv', comment='#')
tension_evolution = pd.read_csv('data/tension_evolution.csv', comment='#')
h0_compilation = pd.read_csv('data/h0_measurements_compilation.csv', comment='#')
cchp_crossval_summary = pd.read_csv('data/cchp_crossval_summary.csv', comment='#')
trgb_cepheid = pd.read_csv('data/cchp_trgb_cepheid_comparison.csv', comment='#')
trgb_jagb = pd.read_csv('data/cchp_trgb_jagb_comparison.csv', comment='#')

print("✓ Data files loaded successfully\n")
print(f"  Systematic budget: {len(systematic_budget)} error sources")
print(f"  Tension evolution: {len(tension_evolution)} stages")
print(f"  H₀ compilation: {len(h0_compilation)} measurements")
print(f"  JWST Cepheid-TRGB: {len(trgb_cepheid)} galaxies")
print(f"  JWST JAGB-TRGB: {len(trgb_jagb)} galaxies")

---

## Key Result 1: Systematic Error Underestimation

**Claim:** SH0ES underestimates systematic uncertainties by **1.6×**

- **SH0ES assessment:** σ_sys = 1.04 km/s/Mpc (uncorrelated)
- **Our assessment:** σ_sys = 1.71 km/s/Mpc (with realistic correlations)
- **Underestimation factor:** 1.6× when accounting for correlated error sources

In [ ]:
# Display systematic error budget
print("Systematic Error Budget Comparison")
print("=" * 80)
display(systematic_budget[['Error_Source', 'SH0ES_Estimate_km_s_Mpc', 'Our_Assessment_km_s_Mpc', 'Confidence_Level']].head(10))

# Calculate totals
systematic_only = systematic_budget[systematic_budget['Error_Source'] != 'Statistical_Uncertainty']
shoes_sys = np.sqrt(np.sum(systematic_only['SH0ES_Estimate_km_s_Mpc']**2))
our_sys_uncorr = np.sqrt(np.sum(systematic_only['Our_Assessment_km_s_Mpc']**2))
our_sys_corr = 1.71  # From correlation matrix calculation

print("\nSystematic Uncertainty Totals:")
print(f"  SH0ES (uncorrelated):      σ_sys = {shoes_sys:.2f} km/s/Mpc")
print(f"  Our assessment (uncorr):   σ_sys = {our_sys_uncorr:.2f} km/s/Mpc")
print(f"  Our assessment (corr):     σ_sys = {our_sys_corr:.2f} km/s/Mpc")
print(f"\n  Underestimation factor (uncorrelated): {our_sys_uncorr/shoes_sys:.2f}×")
print(f"  Underestimation factor (correlated):   {our_sys_corr/shoes_sys:.2f}×")

### Visualize Systematic Budget Comparison

In [ ]:
# Create comparison bar chart
fig, ax = plt.subplots(figsize=(12, 6))

# Prepare data (exclude statistical uncertainty)
plot_data = systematic_only.copy()
x = np.arange(len(plot_data))
width = 0.35

# Create bars
bars1 = ax.bar(x - width/2, plot_data['SH0ES_Estimate_km_s_Mpc'], width, 
               label='SH0ES Assessment', color='#3498db', alpha=0.8)
bars2 = ax.bar(x + width/2, plot_data['Our_Assessment_km_s_Mpc'], width, 
               label='Our Assessment', color='#e74c3c', alpha=0.8)

# Customize plot
ax.set_xlabel('Systematic Error Source', fontsize=12, fontweight='bold')
ax.set_ylabel('Uncertainty (km/s/Mpc)', fontsize=12, fontweight='bold')
ax.set_title('Systematic Error Budget: SH0ES vs. Our Assessment', fontsize=14, fontweight='bold', pad=20)
ax.set_xticks(x)
ax.set_xticklabels(plot_data['Error_Source'].str.replace('_', ' '), rotation=45, ha='right')
ax.legend(fontsize=11, loc='upper right')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✓ Figure: Systematic error budget comparison")

---

## Key Result 2: Tension Reduction Through Five Stages

**Claim:** Realistic systematics reduce tension from **5.9σ → 1.1σ**

Progressive reduction through:
1. **Stage 1:** Statistical only → 5.9σ
2. **Stage 2:** Add uncorrelated systematics → 4.0σ
3. **Stage 3:** Adopt Scenario A parallax → 4.0σ
4. **Stage 4:** Apply period distribution correction → 1.9σ
5. **Stage 5:** Add metallicity correction + realistic correlations → 1.1σ

In [ ]:
# Display tension evolution table
print("Tension Evolution: Five Progressive Stages")
print("=" * 80)
display(tension_evolution[['Stage', 'H0_km_s_Mpc', 'Sigma_km_s_Mpc', 'Tension_sigma', 'Description']])

# Calculate tension reduction
initial_tension = tension_evolution.iloc[0]['Tension_sigma']
final_tension = tension_evolution.iloc[-1]['Tension_sigma']
reduction_factor = initial_tension / final_tension

print(f"\nTension Reduction Summary:")
print(f"  Initial (Stage 1):  {initial_tension:.1f}σ")
print(f"  Final (Stage 5):    {final_tension:.1f}σ")
print(f"  Reduction factor:   {reduction_factor:.1f}×")

### Visualize Tension Evolution

In [ ]:
# Create tension evolution plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Left panel: H₀ values with error bars
stages = range(1, len(tension_evolution) + 1)
h0_values = tension_evolution['H0_km_s_Mpc']
sigma_values = tension_evolution['Sigma_km_s_Mpc']

ax1.errorbar(stages, h0_values, yerr=sigma_values, 
             marker='o', markersize=10, linewidth=2, capsize=5,
             color='#e74c3c', label='Corrected H₀', zorder=3)
ax1.axhline(y=67.36, color='#3498db', linestyle='--', linewidth=2, 
            label='Planck CMB (67.36 ± 0.54)', zorder=2)
ax1.fill_between([0.5, 5.5], 67.36-0.54, 67.36+0.54, 
                  color='#3498db', alpha=0.2, zorder=1)

ax1.set_xlabel('Stage', fontsize=12, fontweight='bold')
ax1.set_ylabel('H₀ (km/s/Mpc)', fontsize=12, fontweight='bold')
ax1.set_title('H₀ Evolution Through Systematic Corrections', fontsize=13, fontweight='bold')
ax1.set_xticks(stages)
ax1.legend(fontsize=10, loc='upper right')
ax1.grid(alpha=0.3)
ax1.set_xlim(0.5, 5.5)

# Right panel: Tension reduction
tension_values = tension_evolution['Tension_sigma']
colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(stages)))

bars = ax2.bar(stages, tension_values, color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)
ax2.axhline(y=3, color='orange', linestyle='--', linewidth=2, 
            label='3σ threshold', alpha=0.7)
ax2.axhline(y=2, color='green', linestyle='--', linewidth=2, 
            label='2σ threshold', alpha=0.7)

# Add value labels on bars
for i, (stage, tension) in enumerate(zip(stages, tension_values)):
    ax2.text(stage, tension + 0.2, f'{tension:.1f}σ', 
             ha='center', va='bottom', fontweight='bold', fontsize=11)

ax2.set_xlabel('Stage', fontsize=12, fontweight='bold')
ax2.set_ylabel('Tension vs Planck (σ)', fontsize=12, fontweight='bold')
ax2.set_title('Hubble Tension Reduction', fontsize=13, fontweight='bold')
ax2.set_xticks(stages)
ax2.legend(fontsize=10, loc='upper right')
ax2.grid(axis='y', alpha=0.3)
ax2.set_xlim(0.5, 5.5)
ax2.set_ylim(0, 7)

plt.tight_layout()
plt.show()

print("\n✓ Figure: Tension evolution through five stages")

---

## Key Result 3: JWST Cross-Validation Evidence

**Claim:** JWST data confirms systematic underestimation

- **TRGB-JAGB agreement:** RMS ≈ 0.048 mag (baseline precision)
- **Cepheid-TRGB scatter:** RMS ≈ 0.108 mag
- **Excess scatter ratio:** 2.3× larger for Cepheids

This provides direct observational evidence for enlarged Cepheid systematic uncertainties.

In [ ]:
# Display JWST cross-validation statistics
print("JWST NIRCam Cross-Validation Summary")
print("=" * 80)
display(cchp_crossval_summary)

# Extract key statistics
jagb_trgb_rms = cchp_crossval_summary[cchp_crossval_summary['Comparison'] == 'JAGB vs TRGB']['RMS_Scatter_mag'].values[0]
cep_trgb_rms = cchp_crossval_summary[cchp_crossval_summary['Comparison'] == 'Cepheid vs TRGB']['RMS_Scatter_mag'].values[0]
scatter_ratio = cep_trgb_rms / jagb_trgb_rms

print(f"\nKey Statistics:")
print(f"  JAGB-TRGB RMS:        {jagb_trgb_rms:.3f} mag  (baseline precision)")
print(f"  Cepheid-TRGB RMS:     {cep_trgb_rms:.3f} mag  (excess scatter)")
print(f"  Scatter ratio:        {scatter_ratio:.2f}×")
print(f"\n  Interpretation: Cepheid distances show {scatter_ratio:.1f}× larger scatter")
print(f"                  than the JWST baseline, confirming enlarged systematics.")

### Visualize JWST Cross-Validation

In [ ]:
# Create side-by-side scatter plots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Left panel: JAGB vs TRGB (baseline)
ax1.errorbar(trgb_jagb['mu_TRGB'], trgb_jagb['mu_JAGB'],
             xerr=trgb_jagb['sigma_TRGB'], yerr=trgb_jagb['sigma_JAGB'],
             fmt='o', markersize=8, color='#27ae60', alpha=0.7,
             elinewidth=1.5, capsize=3, label='JAGB vs TRGB')

# Add 1:1 line
lim_min = min(trgb_jagb['mu_TRGB'].min(), trgb_jagb['mu_JAGB'].min()) - 0.1
lim_max = max(trgb_jagb['mu_TRGB'].max(), trgb_jagb['mu_JAGB'].max()) + 0.1
ax1.plot([lim_min, lim_max], [lim_min, lim_max], 'k--', linewidth=2, alpha=0.5, label='1:1 line')

ax1.set_xlabel('μ (TRGB) [mag]', fontsize=12, fontweight='bold')
ax1.set_ylabel('μ (JAGB) [mag]', fontsize=12, fontweight='bold')
ax1.set_title(f'JWST Baseline: JAGB vs TRGB\nRMS = {jagb_trgb_rms:.3f} mag', 
              fontsize=13, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(alpha=0.3)
ax1.set_aspect('equal')

# Right panel: Cepheid vs TRGB (excess scatter)
ax2.errorbar(trgb_cepheid['mu_TRGB_CCHP'], trgb_cepheid['mu_Cepheid_R22'],
             xerr=trgb_cepheid['sigma_TRGB'], yerr=trgb_cepheid['sigma_Cepheid'],
             fmt='o', markersize=8, color='#e74c3c', alpha=0.7,
             elinewidth=1.5, capsize=3, label='Cepheid vs TRGB')

# Add 1:1 line
lim_min = min(trgb_cepheid['mu_TRGB_CCHP'].min(), trgb_cepheid['mu_Cepheid_R22'].min()) - 0.2
lim_max = max(trgb_cepheid['mu_TRGB_CCHP'].max(), trgb_cepheid['mu_Cepheid_R22'].max()) + 0.2
ax2.plot([lim_min, lim_max], [lim_min, lim_max], 'k--', linewidth=2, alpha=0.5, label='1:1 line')

ax2.set_xlabel('μ (TRGB) [mag]', fontsize=12, fontweight='bold')
ax2.set_ylabel('μ (Cepheid) [mag]', fontsize=12, fontweight='bold')
ax2.set_title(f'Excess Cepheid Scatter\nRMS = {cep_trgb_rms:.3f} mag ({scatter_ratio:.1f}× baseline)', 
              fontsize=13, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(alpha=0.3)
ax2.set_aspect('equal')

plt.tight_layout()
plt.show()

print("\n✓ Figure: JWST cross-validation showing 2.3× excess Cepheid scatter")

---

## Key Result 4: Multi-Method Convergence

**Claim:** Independent late-universe methods converge at H₀ ≈ 68 km/s/Mpc

- **Corrected Cepheid:** 69.67 ± 1.89 km/s/Mpc
- **TRGB:** 69.85 ± 2.33 km/s/Mpc
- **Cosmic Chronometers:** 68.33 ± 1.57 km/s/Mpc
- **JAGB:** 67.96 ± 2.65 km/s/Mpc
- **Three-method mean** (JAGB + CC + Planck): 67.48 ± 0.50 km/s/Mpc

All methods are consistent within ~2σ, supporting convergence.

In [ ]:
# Display H₀ compilation
print("H₀ Measurement Compilation")
print("=" * 80)
display(h0_compilation[['Method', 'H0_km_s_Mpc', 'Sigma_km_s_Mpc', 'Category']].sort_values('H0_km_s_Mpc'))

# Calculate statistics
h0_mean = h0_compilation['H0_km_s_Mpc'].mean()
h0_std = h0_compilation['H0_km_s_Mpc'].std()
h0_range = h0_compilation['H0_km_s_Mpc'].max() - h0_compilation['H0_km_s_Mpc'].min()

print(f"\nH₀ Statistics Across All Methods:")
print(f"  Mean:   {h0_mean:.2f} km/s/Mpc")
print(f"  Std:    {h0_std:.2f} km/s/Mpc")
print(f"  Range:  {h0_range:.2f} km/s/Mpc")
print(f"\n  Three-method convergence: {h0_compilation[h0_compilation['Method']=='Weighted Mean']['H0_km_s_Mpc'].values[0]:.2f} ± "
      f"{h0_compilation[h0_compilation['Method']=='Weighted Mean']['Sigma_km_s_Mpc'].values[0]:.2f} km/s/Mpc")

### Visualize Multi-Method H₀ Forest Plot

In [ ]:
# Create forest plot
fig, ax = plt.subplots(figsize=(12, 8))

# Sort by H₀ value
plot_data = h0_compilation.sort_values('H0_km_s_Mpc').copy()
y_positions = np.arange(len(plot_data))

# Color by category
category_colors = {
    'Early Universe': '#3498db',
    'Distance Ladder': '#e74c3c',
    'Model-Independent': '#27ae60',
    'Convergence': '#f39c12'
}

# Plot error bars and points
for i, row in plot_data.iterrows():
    color = category_colors.get(row['Category'], 'gray')
    y = list(plot_data.index).index(i)
    
    ax.errorbar(row['H0_km_s_Mpc'], y, 
                xerr=row['Sigma_km_s_Mpc'],
                fmt='o', markersize=10, color=color, 
                elinewidth=2.5, capsize=5, capthick=2.5,
                label=row['Category'] if row['Category'] not in ax.get_legend_handles_labels()[1] else '',
                alpha=0.8, zorder=3)

# Add weighted mean line
weighted_mean = plot_data[plot_data['Method'] == 'Weighted Mean']['H0_km_s_Mpc'].values[0]
weighted_sigma = plot_data[plot_data['Method'] == 'Weighted Mean']['Sigma_km_s_Mpc'].values[0]
ax.axvline(weighted_mean, color='#f39c12', linestyle='--', linewidth=2.5, 
           label=f'Three-method mean: {weighted_mean:.2f}±{weighted_sigma:.2f}', zorder=2)
ax.axvspan(weighted_mean - weighted_sigma, weighted_mean + weighted_sigma, 
           color='#f39c12', alpha=0.15, zorder=1)

# Customize plot
ax.set_yticks(y_positions)
ax.set_yticklabels(plot_data['Method'])
ax.set_xlabel('H₀ (km/s/Mpc)', fontsize=12, fontweight='bold')
ax.set_ylabel('Method', fontsize=12, fontweight='bold')
ax.set_title('Multi-Method H₀ Compilation: Convergence at ~68 km/s/Mpc', 
             fontsize=14, fontweight='bold', pad=20)
ax.legend(fontsize=10, loc='lower right', framealpha=0.9)
ax.grid(axis='x', alpha=0.3)
ax.set_xlim(65, 75)

plt.tight_layout()
plt.show()

print("\n✓ Figure: H₀ compilation showing multi-method convergence")

---

## Summary: Key Takeaways

This interactive notebook reproduced the four main findings:

### 1. Systematic Underestimation ✓
- SH0ES underestimates systematic uncertainties by **1.6×** when accounting for realistic correlations
- σ_sys increases from 1.04 → 1.71 km/s/Mpc

### 2. Tension Reduction ✓
- Realistic error accounting reduces Hubble tension from **5.9σ → 1.1σ**
- Progressive reduction through five stages of systematic corrections
- **5.4× reduction** in reported tension

### 3. JWST Validation ✓
- Independent JWST observations confirm excess Cepheid scatter
- Cepheid-TRGB RMS **2.3× larger** than TRGB-JAGB baseline
- Direct observational evidence for enlarged Cepheid systematics

### 4. Multi-Method Convergence ✓
- Independent late-universe methods converge at **H₀ ≈ 68 km/s/Mpc**
- Three-method mean: **67.48 ± 0.50 km/s/Mpc**
- Excellent consistency (χ²_red ≈ 0.04) across methods

---

## Conclusion

The Hubble tension is **largely consistent with a measurement artifact** rather than requiring new physics. When realistic systematic uncertainties are properly accounted for, the tension reduces to ~1σ, well within normal statistical fluctuations.

**Citation:**  
Wiley, A. (2025). *Forensic Analysis of Distance Ladder Systematics: The Hubble Tension Reduced from ~6σ to ~1σ*. The Astrophysical Journal (submitted).

**Repository:** https://github.com/ylecoyote/distance-ladder-systematics

---

## Next Steps

To explore the full analysis:

1. **Run all analysis scripts:**
   ```bash
   python analysis/calculate_error_budget.py
   python analysis/calculate_tension_evolution.py
   python analysis/create_manuscript_tables.py
   ```

2. **Generate all figures:**
   ```bash
   python analysis/create_figure1_tension_evolution.py
   python analysis/create_figure2_error_budget.py
   python analysis/create_figure3_cchp_crossval_real.py
   python analysis/create_figure4_h0_compilation.py
   python analysis/create_figure5_hz_fit_intrinsic_scatter.py
   ```

3. **Read the manuscript:** See [`manuscript.tex`](manuscript/manuscript.tex) for full technical details

4. **Explore the data:** All CSV files in `data/` have header comments documenting sources and methods